In [ ]:
import sys
import os
sys.path.append("../src/")

In [ ]:
import importlib
import cdsaxs
import numpy as np

# if you are not actively developing the code, you can comment the rest of these out
importlib.reload(cdsaxs)
importlib.reload(cdsaxs.data1d)
importlib.reload(cdsaxs.data2d)
importlib.reload(cdsaxs.dataset)
importlib.reload(cdsaxs.loaders)
importlib.reload(cdsaxs.metadata)
importlib.reload(cdsaxs.plotting)
importlib.reload(cdsaxs._plotting_tools)
importlib.reload(cdsaxs.reduction)

In [ ]:
# set the path to the metadata CSV file on your computer for an SMI dataset
csv_path = os.path.abspath("C:\\Users\\cmw5\\Documents\\beamtime_data\\20250310_NSLSII_SMI\\data\\combined_W204_F2_singleScan\\W204_metadata.csv")

# currently the General TIFF loader (with csv metadata file) is the only one implemented in the refactored code
dataset = cdsaxs.loaders.GeneralTIFFLoader(csv_path, name="W204 F2")
dataset.update_all_metadata({'sdd_cm': 504.867})

In [ ]:
dataset.datas

In [ ]:
# attributes of the DataQdyQdx class
data = dataset.datas["W204_F2measure1_5.2m_16.1keV_num60_00deg_bpm0.417_id857181_combined.tif"]
data.__dict__.keys()

In [ ]:
data.name

In [ ]:
data.metadata

In [ ]:
# list methods of the DataQdyQdx class
[method for method in dir(data) if method not in data.__dict__.keys() and method[0]!="_"]

In [ ]:
data.plot_data(show_pixels=False, log_scale=True)

In [ ]:
# we can use the peaks in the image to locate the beam center
beam_center = data.find_beam_center_from_peaks([740, 490], size_qdy_px=10, size_qdx_px=600, peak_axis='qdx', peak_params={'distance': 15, 'height': 10, 'prominence': 100}, peak_find_scale='linear')

In [ ]:
# you can update metadata for each image individually or you can update all at once from the dataset level
# here we are updating the beam center for the whole dataset and this information will get stored for each image
dataset.update_all_metadata({'center_px': beam_center})
data.plot_data()

In [ ]:
# we can integrate each image with three different modes
qslice = data.integrate_box_of_size(5, 400, mode='sum', axis='qdy', show_plot=True, log_scale=True)

In [ ]:
qslice = data.integrate_box_of_q_range((-0.001, 0.001), (-0.1, 0.1), mode='sum', axis='qdy', show_plot=True, log_scale=True)

In [ ]:
qslice = data.integrate_box((733, 743), (292, 692), mode='sum', axis='qdy', show_plot=True, log_scale=True)

In [ ]:
output = data.find_peaks1D('box_size', {'size_qdy_px': 5, 'size_qdx_px': 700, 'mode': 'sum', 'axis': 'qdy'}, {'distance': 15, 'height': 10, 'prominence': 100}, peak_find_scale='linear')

In [ ]:
output

In [ ]:
# once we know the integration box we would like to use, we can apply it to the whole dataset and generate an integrated dataset (collection of integrated q slices)
cdsaxs.reduction.integrate_dataset_box_of_size(dataset, 5, 400, mode='sum', axis='qdy', in_place=True)
dataset.plot_integrated_dataset()

In [ ]:
cdsaxs.reduction.create_reduced_QszQsx(dataset, integrated_index=0)
dataset.plot_reduced_dataset()

In [ ]:
q_values = np.arange(1, 7)*2*np.pi/1000
cdsaxs.reduction.slice_reduced_dataset(dataset, q_values=list(q_values)+list(-1*q_values), q_widths=0.001, in_place=True, show_plot=True)

In [ ]:
dataset.reduced_slices

In [ ]:
cdsaxs.plotting.plot_reduced_slices(dataset)

In [ ]:
dataset.plot_reduced_slices()